# 01 — Experiment Exploration via Databricks SDK

**Purpose:** Explore all MLflow experiments in the connected Azure Databricks workspace using the `databricks-sdk` `WorkspaceClient` — the same client used by the FastAPI backend.

**What you'll learn:**
- How to authenticate to a Databricks workspace from a notebook
- How to list and inspect MLflow experiments via the SDK (not `mlflow` client directly)
- How to inspect experiment tags, artifact locations, and lifecycle stage
- How to filter/sort experiments for exploratory data science

**Prerequisites:**
- `databricks-sdk` installed: `pip install databricks-sdk`
- A `.env` file (or environment variables) with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

---

## 1. Environment Setup

Load credentials from `.env` using `python-dotenv`. Never hard-code tokens in notebooks.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load from ../backend/.env (adjust path if running from a different location)
env_path = Path('..') / 'backend' / '.env'
loaded = load_dotenv(dotenv_path=env_path)
print(f"[.env] Loaded: {loaded} (from {env_path.resolve()})")

DATABRICKS_HOST  = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN = os.getenv('DATABRICKS_TOKEN')

if not DATABRICKS_HOST or not DATABRICKS_TOKEN:
    print("WARNING: DATABRICKS_HOST or DATABRICKS_TOKEN not set — mock data will be used below")
else:
    print(f"Databricks host: {DATABRICKS_HOST}")

## 2. Initialise the WorkspaceClient

The `WorkspaceClient` is the central entry point for all Databricks SDK operations.
This mirrors exactly how `core/databricks_client.py` works in the FastAPI backend.

In [ ]:
USE_MOCK = not (DATABRICKS_HOST and DATABRICKS_TOKEN)

if not USE_MOCK:
    from databricks.sdk import WorkspaceClient
    client = WorkspaceClient(
        host=DATABRICKS_HOST,
        token=DATABRICKS_TOKEN,
    )
    print(f"WorkspaceClient initialised for: {DATABRICKS_HOST}")
else:
    client = None
    print("Running in MOCK mode (no Databricks credentials found)")

## 3. List All Experiments

Using `client.experiments.list()` — equivalent to `mlflow.search_experiments()` but via the SDK.

In [ ]:
from datetime import datetime, timezone

# --- Mock data (used when no Databricks connection) ---
MOCK_EXPERIMENTS = [
    {
        "experiment_id": "1",
        "name": "drug_efficacy_xgboost_v1",
        "artifact_location": "dbfs:/mlflow/experiments/1",
        "lifecycle_stage": "active",
        "tags": {"team": "commercial_ai", "market": "UK", "brand": "Tagrisso"},
        "creation_time_ms": 1743156000000,
    },
    {
        "experiment_id": "2",
        "name": "drug_efficacy_lightgbm_v2",
        "artifact_location": "dbfs:/mlflow/experiments/2",
        "lifecycle_stage": "active",
        "tags": {"team": "commercial_ai", "market": "DE", "brand": "Imfinzi"},
        "creation_time_ms": 1743242400000,
    },
    {
        "experiment_id": "3",
        "name": "patient_segmentation_kmeans_v1",
        "artifact_location": "dbfs:/mlflow/experiments/3",
        "lifecycle_stage": "active",
        "tags": {"team": "medical_ai", "market": "US"},
        "creation_time_ms": 1743328800000,
    },
]


def ms_to_dt(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


if client is not None:
    # Real Databricks call
    experiments_raw = list(client.experiments.list())
    experiments = [
        {
            "experiment_id": str(e.experiment_id),
            "name": e.name,
            "artifact_location": e.artifact_location,
            "lifecycle_stage": e.lifecycle_stage,
            "tags": {t.key: t.value for t in (e.tags or [])},
            "creation_time_ms": e.creation_time,
        }
        for e in experiments_raw
    ]
else:
    experiments = MOCK_EXPERIMENTS

print(f"Found {len(experiments)} experiment(s)\n")
for exp in experiments:
    created = ms_to_dt(exp['creation_time_ms']) if exp.get('creation_time_ms') else 'N/A'
    print(f"  [{exp['experiment_id']}] {exp['name']} | {exp['lifecycle_stage']} | created: {created}")

## 4. Inspect a Specific Experiment

Fetch full detail for one experiment: tags, artifact path, and lifecycle stage.

In [ ]:
import json

TARGET_EXPERIMENT_ID = "1"  # Change this to explore other experiments

if client is not None:
    result = client.experiments.get(experiment_id=TARGET_EXPERIMENT_ID)
    exp = result.experiment
    detail = {
        "experiment_id": str(exp.experiment_id),
        "name": exp.name,
        "artifact_location": exp.artifact_location,
        "lifecycle_stage": exp.lifecycle_stage,
        "tags": {t.key: t.value for t in (exp.tags or [])},
    }
else:
    detail = next(e for e in MOCK_EXPERIMENTS if e["experiment_id"] == TARGET_EXPERIMENT_ID)

print(json.dumps(detail, indent=2, default=str))

## 5. Filter Active Experiments

Demonstrate filtering by `lifecycle_stage` and sorting by creation time.

In [ ]:
active_experiments = [
    e for e in experiments
    if e.get('lifecycle_stage', 'active') == 'active'
]

print(f"Active experiments: {len(active_experiments)} / {len(experiments)} total\n")
for exp in active_experiments:
    tags_display = ", ".join(f"{k}={v}" for k, v in (exp.get('tags') or {}).items())
    print(f"  {exp['name']}")
    print(f"    tags: {tags_display or 'none'}")
    print(f"    artifact_location: {exp.get('artifact_location', 'N/A')}")
    print()

## 6. Summary

This notebook demonstrated:
- Authentication to Databricks via `WorkspaceClient` (same pattern as the FastAPI backend)
- Listing and inspecting experiments using `client.experiments.list()` and `client.experiments.get()`
- Graceful fallback to mock data when credentials are absent

**Next:** See `02_run_comparison.ipynb` to explore and compare runs within an experiment.